In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json

In [ ]:
# Load in site water level data dictionary
with open('/capstone/aridgw/data/site_data/site_coords.json', 'r') as f:
    site_coords = {k: tuple(v) for k, v in json.load(f).items()}

# Load in clean water level data csv that will be merged to
clean_site_data = "/capstone/aridgw/data/site_data/clean_site_waterdata.csv"
df_sites = pd.read_csv(clean_site_data)

# Load in clean water level data csv that will be merged to
# clean_et_monthly_data = "/capstone/aridgw/data/site_data/openet_et_monthly_clean.csv"
# df_monthly = pd.read_csv(clean_et_monthly_data)

# df_monthly = df_monthly[~((df_monthly["year"] == 2019) & (df_monthly["version"] == 2.1))]

In [ ]:

# Use API to access OpenET Data
header = {"Authorization": "31WmFjhG0zzzyqT7LyXfuTS9sHopnzU90kxnLRFSPAuwE8w4N7kg6zuPeBRs"}

# Set sites query parameter to site coordinates dictionary
sites = site_coords

# Select boundary box size
size_km = 1

# Extract data sections less than 10 years as required by the API
date_ranges = [
    ("2000-01-01", "2009-12-31", 2.0), 
    ("2010-01-01", "2019-12-31", 2.0),
    ("2020-01-01", "2025-12-31", 2.1), # Use version 2.1 for newer versions which may not have accurate data in older versions
]

dfs = [] # Create empty dataframe

# Create function to generate boundary box given coordinates and boundary box size
def make_square_geometry(lat, lon, size_km=1):
    half = size_km / 2
    lat_offset = half / 111.0
    lon_offset = half / (111.0 * np.cos(np.radians(lat)))
    return [
        lon - lon_offset, lat + lat_offset,
        lon - lon_offset, lat - lat_offset,
        lon + lon_offset, lat - lat_offset,
        lon + lon_offset, lat + lat_offset
    ]

# Use API to extract ET data 
for name, (lat, lon) in sites.items():
    print(f"Querying {name}...")
    
    for start_date, end_date, version in date_ranges:

        polygon_args = { # Define query paramters
            "date_range": [start_date, end_date],
            "interval": "monthly",
            "geometry": make_square_geometry(lat, lon, size_km),
            "model": "Ensemble",
            "reducer": "mean",
            "variable": "ET",
            "reference_et": "gridMET",
            "units": "mm",
            "file_format": "JSON",
            "version": version
        }

        resp = requests.post( # Request polygon API for this extraction
            headers=header,
            json=polygon_args,
            url="https://openet-api.org/raster/timeseries/polygon"
        )

        print(f"  {name} {start_date}–{end_date} — Status: {resp.status_code}") # Report status of data to console

        if resp.status_code != 200: # Return error if unable to extract data
            print(f"  ERROR: {resp.json()}")
            continue

        data = resp.json()
        df = pd.DataFrame(data)

        # Add columns to dataframe to specify site specific information
        df["site"] = name
        df["latitude"] = lat
        df["longitude"] = lon
        df["radius_km"] = size_km
        df["version"] = version
        dfs.append(df)

df_monthly = pd.concat(dfs, ignore_index=True) # Add new data as new observations in dataframe
print(df_monthly.head())

Querying KSGS.371852100505801...
  KSGS.371852100505801 2000-01-01–2009-12-31 — Status: 200
  KSGS.371852100505801 2010-01-01–2019-12-31 — Status: 200
  KSGS.371852100505801 2019-01-01–2025-12-31 — Status: 200
Querying KSGS.372043101363101...
  KSGS.372043101363101 2000-01-01–2009-12-31 — Status: 200
  KSGS.372043101363101 2010-01-01–2019-12-31 — Status: 200
  KSGS.372043101363101 2019-01-01–2025-12-31 — Status: 200
Querying KSGS.372539100142504...
  KSGS.372539100142504 2000-01-01–2009-12-31 — Status: 200
  KSGS.372539100142504 2010-01-01–2019-12-31 — Status: 200
  KSGS.372539100142504 2019-01-01–2025-12-31 — Status: 200
Querying KSGS.373331098033301...
  KSGS.373331098033301 2000-01-01–2009-12-31 — Status: 200
  KSGS.373331098033301 2010-01-01–2019-12-31 — Status: 200
  KSGS.373331098033301 2019-01-01–2025-12-31 — Status: 200
Querying KSGS.373607100565301...
  KSGS.373607100565301 2000-01-01–2009-12-31 — Status: 200
  KSGS.373607100565301 2010-01-01–2019-12-31 — Status: 200
  KSGS.37

In [17]:
# Convert time column to datetime, extract year
df_monthly["time"] = pd.to_datetime(df_monthly["time"])
df_monthly["year"] = df_monthly["time"].dt.year
df_monthly

,time,et,site,latitude,longitude,radius_km,version,year
0,2000-01-01,12.573,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
1,2000-02-01,27.326,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
2,2000-03-01,40.734,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
3,2000-04-01,55.314,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
4,2000-05-01,104.492,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
...,...,...,...,...,...,...,...,...
16531,2025-08-01,70.545,southern_willcox,31.36597,-109.6630,1,2.1,2025
16532,2025-09-01,44.656,southern_willcox,31.36597,-109.6630,1,2.1,2025
16533,2025-10-01,47.854,southern_willcox,31.36597,-109.6630,1,2.1,2025
16534,2025-11-01,27.207,southern_willcox,31.36597,-109.6630,1,2.1,2025


In [35]:
# Convert monthly ET data into csv
df_monthly.to_csv("openet_data_monthly.csv", index=False)

In [36]:
# Check number of monthly et observations per site 
df_monthly.groupby(["site"]).size().reset_index(name="n") 

,site,n
0,KSGS.371852100505801,312
1,KSGS.372043101363101,312
2,KSGS.372539100142504,312
3,KSGS.373331098033301,312
4,KSGS.373607100565301,312
5,KSGS.374111099070401,312
6,KSGS.374125100344101,312
7,KSGS.374747100552101,312
8,KSGS.375145100485701,312
9,KSGS.375454101075401,312


In [37]:
# Calculate annual sum of monthly ET data 
df_annual = df_monthly.groupby(['site', 'year', 'version', 'radius_km'])['et'].sum().reset_index()

In [38]:
# Convert annual ET data into csv
df_annual.to_csv("openet_data_annual.csv", index=False)

In [39]:
# Check number of annual et observations per site 
df_annual.groupby(["site"]).size().reset_index(name="n") 

,site,n
0,KSGS.371852100505801,26
1,KSGS.372043101363101,26
2,KSGS.372539100142504,26
3,KSGS.373331098033301,26
4,KSGS.373607100565301,26
5,KSGS.374111099070401,26
6,KSGS.374125100344101,26
7,KSGS.374747100552101,26
8,KSGS.375145100485701,26
9,KSGS.375454101075401,26


In [40]:
# Merge ET data to water level data 
df_merged_et = pd.merge(
    df_sites,
    df_annual,
    left_on=['site_id', 'year_value'],
    right_on=['site', 'year'],
    how='outer' # Keep all rows for both data frames
)

In [41]:
# Check number of observations per site in merged df
df_merged_et.groupby(["site_id"]).size().reset_index(name="n")

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [ ]:
# Turn merged df into csv
df_merged_et.to_csv("merged_openet_data.csv", index=False)